In [ ]:
from pathlib import Path
from time import time

import jax
import numpy
import orbax
import torch
from flax.training import early_stopping, orbax_utils
from tqdm import tqdm
from utils import general_params, get_arg, get_folders_dbb, search_folder

from architecture import (
    EVAL_FUNCTION,
    UPDATE_FUNCTION,
    MyDataloader,
    create_train_state,
    load_state,
    range_compression,
)

In [ ]:
ONGOING = "Resnet-10_batch-32_epoch-100_norma-compress_dbb-v7_prelayer-64_reslayer-2222_momemtum-09_kernel-3_pad-1_modepad-constant_dist-uni"  # noqa: E501
BEST_MODEL_FOLDER = Path(search_folder(name=f"RESULTS/{ONGOING}"))
MODEL, STATE = load_state(BEST_MODEL_FOLDER)
MODEL_CONFIG = STATE["model_config"]

In [ ]:
VERSION = get_arg(ONGOING, "dbb")  # A CHANGER
if VERSION == "v3":
    NB_OF_TRAIN_IMAGES = 40000
    NB_OF_VAL_IMAGES = 5000
if VERSION == "v4":
    NB_OF_TRAIN_IMAGES = 50400
    NB_OF_VAL_IMAGES = 4800
if VERSION == "v7":
    NB_OF_TRAIN_IMAGES = 8000
    NB_OF_VAL_IMAGES = 1000

In [ ]:
if int(MODEL_CONFIG["type_resnet"]) == 10:
    MODEL_CONFIG["type_resnet"] = 20
elif int(MODEL_CONFIG["type_resnet"]) == 11:
    MODEL_CONFIG["type_resnet"] = 21

pretrained_config = {
    "batch_size": int(MODEL_CONFIG["batch_size"]),
    "dbb_version": int(MODEL_CONFIG["dbb_version"]),
    "kernel_size": int(MODEL_CONFIG["kernel_size"]),
    "min_shears": float(MODEL_CONFIG["min_shears"]),
    "max_shears": float(MODEL_CONFIG["max_shears"]),
    "mean_images": float(MODEL_CONFIG["mean_images"]),
    "mode_pad": "constant",
    "momentum": float(MODEL_CONFIG["momentum"]),
    "normalisation": int(MODEL_CONFIG["normalisation"]),
    "number_of_epochs": int(MODEL_CONFIG["number_of_epochs"]),
    "pad": int(MODEL_CONFIG["pad"]),
    "pre_layer": [int(x) for x in MODEL_CONFIG["pre_layer"]],
    "type_resnet": int(MODEL_CONFIG["type_resnet"]),
    "std_images": float(MODEL_CONFIG["std_images"]),
    "resnet_layer": [int(x) for x in MODEL_CONFIG["resnet_layer"]],
    "shear_bounds": numpy.array([1e-5, 1 - 1e-5]),
}


pretrained_config

In [ ]:
BATCH_SIZE = int(MODEL_CONFIG["batch_size"])
NB_BATCH_TRAIN = NB_OF_TRAIN_IMAGES // BATCH_SIZE + int(
    NB_OF_TRAIN_IMAGES % BATCH_SIZE != 0
)
NB_BATCH_VAL = NB_OF_VAL_IMAGES // BATCH_SIZE + int(NB_OF_VAL_IMAGES % BATCH_SIZE != 0)

In [ ]:
schedule, state, model_config = create_train_state(
    nb_batch_train=NB_BATCH_TRAIN, model_config=pretrained_config, pretrained=STATE
)

In [ ]:
FOLDER_DATASET, _ = general_params()
FOLDER_TRAIN, FOLDER_VAL, _ = get_folders_dbb(FOLDER_DATASET, VERSION)

dataloader_config = {
    "min_shears": float(MODEL_CONFIG["min_shears"]),
    "max_shears": float(MODEL_CONFIG["max_shears"]),
    "mean_images": float(MODEL_CONFIG["mean_images"]),
    "std_images": float(MODEL_CONFIG["std_images"]),
}

train_dataset = MyDataloader(
    data_dir=FOLDER_TRAIN, **dataloader_config, transform=range_compression
)
val_dataset = MyDataloader(
    data_dir=FOLDER_VAL, **dataloader_config, transform=range_compression
)

NUM_WORKERS = 1
PREFETCH = 2
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    pin_memory=False,
    prefetch_factor=PREFETCH,
    drop_last=True,
)
val_loader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=PREFETCH,
    drop_last=True,
)

In [ ]:
# TRAINING INIT
type_resnet = model_config["type_resnet"]
list_avg_train_losses, list_avg_val_losses = [], []
CURRENT_UPDATE_FUNCTION = UPDATE_FUNCTION[type_resnet]
CURRENT_EVAL_FUNCTION = EVAL_FUNCTION[type_resnet]
early_stop = early_stopping.EarlyStopping(min_delta=1e-10, patience=50)

In [ ]:
NUMBER_OF_EPOCHS = 50
FOLDER_NEW_RESULTS = BEST_MODEL_FOLDER.parent / (
    BEST_MODEL_FOLDER.name + f"_FINETUNED-{NUMBER_OF_EPOCHS}"
)
FOLDER_NEW_RESULTS.mkdir(parents=True, exist_ok=True)

for epoch in range(NUMBER_OF_EPOCHS):
    init_time = time()
    list_train_losses, list_val_losses = [], []

    # training dataset
    for x, y in tqdm(train_loader, total=NB_BATCH_TRAIN, desc="tra"):
        state, loss = CURRENT_UPDATE_FUNCTION(
            state, jax.numpy.array(x), jax.numpy.array(y)
        )
        list_train_losses.append(loss)

    # validation dataset
    for x, y in tqdm(val_loader, total=NB_BATCH_VAL, desc="val"):
        val_loss = CURRENT_EVAL_FUNCTION(state, jax.numpy.array(x), jax.numpy.array(y))
        list_val_losses.append(val_loss)

    # savings losses
    train_losses = jax.numpy.mean(jax.numpy.stack(list_train_losses))
    list_avg_train_losses.append(train_losses)
    val_losses = jax.numpy.mean(jax.numpy.stack(list_val_losses))
    list_avg_val_losses.append(val_losses)
    print(f"epoch {epoch + 1} loss: train {train_losses:.2e} loss val {val_losses:.2e}")

    # early stopping & saves
    early_stop = early_stop.update(metric=val_losses)
    if early_stop.should_stop:
        print(f"EARLY STOPPING at epoch {epoch}")
        break
    if early_stop.has_improved:
        print("IMPROVED")
        orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
        config = {"dimensions": numpy.array([5, 3])}
        ckpt = {"model": state, "config": config}
        save_args = orbax_utils.save_args_from_target(ckpt)
        orbax_checkpointer.save(
            FOLDER_NEW_RESULTS, ckpt, save_args=save_args, force=True
        )

    end_time = time() - init_time
    print(f" epoch in {end_time}")

# LOSS CURVES SAVE
numpy.save(FOLDER_NEW_RESULTS / "list_avg_val_losses.npy", list_avg_val_losses)
numpy.save(FOLDER_NEW_RESULTS / "list_avg_train_losses.npy", list_avg_train_losses)

In [ ]:
aa = STATE["params"]["Encoder"]["BasicBlockPool_0"]["Conv_0"]["kernel"]
bb = state.params["Encoder"]["BasicBlockPool_0"]["Conv_0"]["kernel"]
print((aa - bb).all() == 0)

In [ ]:
cc = STATE["params"]["UnivariateNormal"]["Dense_0"]["kernel"]
dd = state.params["UnivariateNormal"]["Dense_0"]["kernel"]
print((cc - dd).all() == 0)